In [25]:
!pip install tf-keras # instala compatibilidade com keras antigo

In [26]:
import cv2 # visão computacional
import numpy as np # cálculos com arrays
import time # medir tempo de processamento por frame
import zipfile # abrir arquivos compactados
import os # manipulação de arquivos e pastas
import tf_keras as keras # carrega o keras compatível

In [27]:
from google.colab import drive
drive.mount('/content/gdrive') # conecta o google drive ao colab

path = "/content/gdrive/MyDrive/Material.zip"
with zipfile.ZipFile(file=path, mode="r") as zip_object: # abre e já fecha automaticamente
    zip_object.extractall("./") # extrai tudo na pasta atual

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [28]:
# carrega a rede neural já treinada no dataset FER2013
model = keras.models.load_model("Material/modelo_02_expressoes.h5", compile=False)

In [29]:
detector_face = cv2.CascadeClassifier('Material/haarcascade_frontalface_default.xml') # carrega o detector de faces

arquivo_video = '/content/supernatural.mp4' # caminho do seu vídeo
cap = cv2.VideoCapture(arquivo_video) # abre o vídeo para leitura

if not cap.isOpened(): # verifica se o vídeo abriu corretamente
    print(f"{arquivo_video}")
else:
    conectado, video = cap.read() # lê o primeiro frame pra pegar as dimensões

    if not conectado or video is None:
        print("so pra confirmar se deu certo")
    else:
        # redimensiona se o vídeo for muito grande pra não travar
        largura_maxima = 600
        if video.shape[1] > largura_maxima:
            proporcao = video.shape[1] / video.shape[0]
            video_largura = largura_maxima
            video_altura = int(video_largura / proporcao)
        else:
            video_largura = video.shape[1]
            video_altura = video.shape[0]

        print(f"resolução: {video_largura}x{video_altura}")

        fourcc = cv2.VideoWriter_fourcc(*'MP4V') # codec pra salvar em mp4
        saida_video = cv2.VideoWriter('supernatural_emocoes.mp4', fourcc, 24, (video_largura, video_altura)) # arquivo de saída

        expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpresa", "Neutro"] # 7 emoções do modelo

        cap.set(cv2.CAP_PROP_POS_FRAMES, 0) # volta pro início do vídeo

resolução: 576x768


In [30]:
frame_count = 0 # contador de frames processados

while (cv2.waitKey(1) < 0): # loop que roda enquanto o vídeo tiver frames
    conectado, frame = cap.read() # lê o próximo frame
    if not conectado: # para quando o vídeo acabar
        break

    t = time.time() # marca o tempo de início desse frame
    frame_count += 1

    # redimensiona o frame se necessário
    if video_largura != frame.shape[1]:
        frame = cv2.resize(frame, (video_largura, video_altura))

    frame_cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) # converte pra cinza

    # detecta rostos com parâmetros mais rigorosos
    faces = detector_face.detectMultiScale(
        frame_cinza,
        scaleFactor=1.1,
        minNeighbors=10, # maior para sre mais preciso
        minSize=(60, 60)  # ignora detecções menores
    )

    if len(faces) > 0: # se encontrou algum rosto no frame
        for (x, y, w, h) in faces: # percorre cada rosto encontrado
            roi = frame_cinza[y:y + h, x:x + w] # recorta só a região do rosto
            roi = cv2.resize(roi, (48, 48)) # redimensiona pro tamanho que o modelo espera
            roi = roi.astype("float") / 255.0 # normaliza os pixels
            roi = np.expand_dims(roi, axis=0) # adiciona dimensão extra que o modelo exige

            result = model.predict(roi, verbose=0)[0] # roda as probabilidades
            confianca = np.max(result) # pega a maior probabilidade

            if confianca > 0.50: # só desenha se tiver pelo menos 50% de certeza
                resultado = np.argmax(result) # índice da emoção com maior probabilidade
                label = f"{expressoes[resultado]} ({confianca*100:.0f}%)" # texto com emoção e confiança

                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2) # retângulo verde ao redor do rosto
                cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA) # escreve a emoção

    # mostra o tempo de processamento
    tempo = time.time() - t
    cv2.putText(frame, f"Frame {frame_count} | {tempo:.2f}s", (10, video_altura - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1, cv2.LINE_AA)

    saida_video.write(frame) # salva o frame processado

saida_video.release() # fecha o arquivo de saída
cap.release() # libera o vídeo de entrada
cv2.destroyAllWindows() # fecha janelas abertas
print(f"Frames: {frame_count}")

Frames: 853
